# Genre Bias in the Academy Awards: A Statistical Analysis of Oscar Winning Films

**Group Members:** Edison Ayran, Diego Inostroza

In [1]:
# libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy.stats import chi2_contingency
from statsmodels.sandbox.stats.runs import runstest_1samp
import statsmodels.api as sm
import statsmodels.formula.api as smf
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix

# Introduction

This project investigates whether the genre of a film influences its probability of winning an Academy Award. The Academy Awards are widely regarded as one of the most prestigious honors in the film industry, and winning an Oscar can significantly impact a film’s cultural recognition, financial success, and the careers of those involved. Because of this influence, it is important to examine whether certain types of films are systematically favored in award outcomes. Using data from the Academy Awards database and the IMDb dataset, this analysis explores whether genres such as Drama or Biography which are often considered “prestige genres,” are more likely to win awards than genres such as Action, Comedy, or Horror. By combining award data with film metadata, this project aims to statistically evaluate whether genre plays a meaningful role in Oscar success.

# Problem Statement

Despite the perception that Academy Awards celebrate the “best” films of the year, critics have long argued that certain genres are consistently favored by voters. In particular, dramatic or biographical films appear to receive disproportionate recognition compared to genres such as comedy, action, or horror. The goal of this project is to investigate whether this perceived genre bias exists using statistical analysis. By examining nomination and winning outcomes across multiple award categories and comparing them with film genre classifications, we aim to determine whether genre significantly affects a film’s probability of winning an Academy Award. Understanding these patterns can provide insight into whether award outcomes reflect purely artistic merit or whether institutional preferences for specific storytelling styles influence the results.

# Objective

Determine whether movie genre significantly influences the probability of winning an Academy
Award across four major film-level categories (Best Picture, Directing, Original Screenplay,
Adapted Screenplay) from 2004–2024, and test whether this relationship has changed
meaningfully between the pre-reform era (2004–2014) and the post-reform era (2015–2024)
using chi-squared tests of independence, the Wald-Wolfowitz runs test, and logistic
regression with genre indicator variables and genre × year interaction terms.

### Research Question

**Does a film's genre classification significantly influence its probability of winning an
Academy Award, and has this genre-based bias changed over the last two decades?**

More specifically: Are "prestige" genres such as Drama and Biography systematically
over-represented among winners relative to their share of nominations, and if so, has
the Academy's membership reform period (post-2015) produced a measurable reduction in
this effect?

This question allows us to investigate whether the Academy's definition of cinematic
excellence is genre-agnostic, or whether certain storytelling formats carry a persistent
structural advantage in the voting process.

# Description of Variables

**From `full_data.csv`:**

| Column              | Type    | Description |
|---------------------|---------|-------------|
| `FilmId`            | string  | IMDb tconst (e.g. `tt0081398`) - primary merge key with IMDb datasets |
| `Film`              | string  | Title of the nominated film |
| `Year`              | string  | Ceremony year (e.g. `2004/05`) |
| `CanonicalCategory` | string  | Standardized category name - filtered to our 4 target categories |
| `Category`          | string  | Raw category label as listed in the official Academy database |
| `Winner`            | boolean | True if the film won; NaN treated as False (nominated but did not win) |
| `Name`              | string  | Person associated with the nomination (director or writer) |

**From `title.basics.tsv.gz`:**

| Column           | Type   | Description |
|------------------|--------|-------------|
| `tconst`         | string | IMDb title identifier - merge key |
| `genres`         | string | Comma-separated genre tags, up to 3 per film (e.g. `Drama,Biography,History`) |
| `runtimeMinutes` | float  | Film runtime in minutes - regression control variable |
| `titleType`      | string | Used to pre-filter to movies only before merging |
| `startYear`      | int    | Release year - used to cross-validate ceremony year |

**From `title.ratings.tsv.gz`:**

| Column          | Type   | Description |
|-----------------|--------|-------------|
| `tconst`        | string | IMDb title identifier - merge key |
| `averageRating` | float  | Weighted average IMDb user rating (scale 1–10) |
| `numVotes`      | int    | Total number of user ratings received |

**Engineered during cleaning:**

| Column          | Type   | Description |
|-----------------|--------|-------------|
| `primary_genre` | string | First genre extracted from `genres` - core categorical predictor in all 3 tests |
| `outcome`       | int    | Binary re-encoding of `Winner` - 1 if won, 0 if nominated only |
| `era`           | int    | 0 = pre-reform (2004–2014), 1 = post-reform (2015–2024) |
| `log_votes`     | float  | Log-transformed `numVotes` to reduce right skew before regression |

## Dataset Composition

Our analysis uses three files merged via IMDb tconst identifiers.

**`full_data.csv` (Primary Oscar Source)**
Contains 12,014 nomination records across all Academy Award categories from 1927–2025.
Each record includes a `FilmId` column with an IMDb tconst (e.g. `tt0081398`),
enabling a clean exact join with IMDb
After filtering to our 4 target categories and ceremony years 2004–2024:

| Category                       | Nominations | Winners |
|--------------------------------|-------------|---------|
| BEST PICTURE                   | 171         | 21      |
| DIRECTING                      | 105         | 21      |
| WRITING (Original Screenplay)  | 105         | 21      |
| WRITING (Adapted Screenplay)   | 105         | 21      |
| **Total**                      | **486**     | **84**  |

**`title.basics.tsv.gz` (IMDb)**
Covers ~10 million IMDb titles. Provides genre classifications (up to 3 per film),
runtime in minutes, title type, and release year. Filtered to `titleType == 'movie'`
before merging.

**`title.ratings.tsv.gz` (IMDb)**
Provides weighted average user rating and total vote count for all rated titles.
Joined to basics on `tconst`, then merged with the Oscar records on `FilmId = tconst`.

Final working dataset after all merges: **486 nomination records**, 10 analytical columns.

## Load the Data ##

In [2]:
# Full Oscar dataset 
oscars_raw = pd.read_csv('full_data.csv', sep='\t', low_memory=False)

# IMDb title.basics 
imdb_basics = pd.read_csv(
    'title.basics.tsv.gz',
    sep='\t',
    compression='gzip',
    low_memory=False,
    na_values='\\N'
)

# IMDb title.ratings 
imdb_ratings = pd.read_csv(
    'title.ratings.tsv.gz',
    sep='\t',
    compression='gzip',
    na_values='\\N'
)

print("Oscar records :", oscars_raw.shape)   
print("IMDb basics   :", imdb_basics.shape)
print("IMDb ratings  :", imdb_ratings.shape)

oscars_raw.head()

Oscar records : (12014, 16)
IMDb basics   : (12347297, 9)
IMDb ratings  : (1642919, 3)


,Ceremony,Year,Class,CanonicalCategory,Category,NomId,Film,FilmId,Name,Nominees,NomineeIds,Winner,Detail,Note,Citation,MultifilmNomination
0,1,1927/28,Acting,ACTOR IN A LEADING ROLE,ACTOR,an0051251,The Noose,tt0019217,Richard Barthelmess,Richard Barthelmess,nm0001932,NaN,Nickie Elkins,NaN,NaN,True
1,1,1927/28,Acting,ACTOR IN A LEADING ROLE,ACTOR,an0051252,The Patent Leather Kid,tt0018253,Richard Barthelmess,Richard Barthelmess,nm0001932,NaN,The Patent Leather Kid,NaN,NaN,True
2,1,1927/28,Acting,ACTOR IN A LEADING ROLE,ACTOR,an0051250a,The Last Command,tt0019071,Emil Jannings,Emil Jannings,nm0417837,True,General Dolgorucki [Grand Duke Sergius Alexander],NaN,NaN,True
3,1,1927/28,Acting,ACTOR IN A LEADING ROLE,ACTOR,an0051250b,The Way of All Flesh,tt0019553,Emil Jannings,Emil Jannings,nm0417837,True,August Schilling,NaN,NaN,True
4,1,1927/28,Acting,ACTRESS IN A LEADING ROLE,ACTRESS,an0051255,A Ship Comes In,tt0018389,Louise Dresser,Louise Dresser,nm0237571,NaN,Mrs. Pleznik,NaN,NaN,NaN


In [3]:
imdb_basics.head()

,tconst,titleType,primaryTitle,originalTitle,isAdult,startYear,endYear,runtimeMinutes,genres
0,tt0000001,short,Carmencita,Carmencita,0,1894.0,NaN,1,"Documentary,Short"
1,tt0000002,short,Le clown et ses chiens,Le clown et ses chiens,0,1892.0,NaN,5,"Animation,Short"
2,tt0000003,short,Poor Pierrot,Pauvre Pierrot,0,1892.0,NaN,5,"Animation,Comedy,Romance"
3,tt0000004,short,Un bon bock,Un bon bock,0,1892.0,NaN,12,"Animation,Short"
4,tt0000005,short,Blacksmith Scene,Blacksmith Scene,0,1893.0,NaN,1,Short


In [4]:
imdb_ratings.head()

,tconst,averageRating,numVotes
0,tt0000001,5.7,2197
1,tt0000002,5.5,309
2,tt0000003,6.5,2302
3,tt0000004,5.1,196
4,tt0000005,6.2,3033


## Merge the Oscars Dataset ##

In [5]:
winning_films = oscars_raw[oscars_raw['Winner'] == True][oscars_raw['Class'].isin(['Title', 'Writing'])]
winning_films
winning_films_with_genre = winning_films.merge(
    imdb_basics[['tconst', 'genres']],
    left_on='FilmId',
    right_on='tconst',
    how='left'
)
winning_films_with_genre = winning_films_with_genre.merge(
    imdb_ratings[['tconst', 'averageRating']],
    on='tconst',
    how='left'
)
winning_films_with_genre

/tmp/ipykernel_350/1322374375.py:1: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  winning_films = oscars_raw[oscars_raw['Winner'] == True][oscars_raw['Class'].isin(['Title', 'Writing'])]


,Ceremony,Year,Class,CanonicalCategory,Category,NomId,Film,FilmId,Name,Nominees,NomineeIds,Winner,Detail,Note,Citation,MultifilmNomination,tconst,genres,averageRating
0,1,1927/28,Title,BEST PICTURE,OUTSTANDING PICTURE,an0051242,Wings,tt0018578,Paramount Famous Lasky,Paramount Famous Lasky,co0023400,True,NaN,NaN,NaN,NaN,tt0018578,"Action,Drama,Romance",7.5
1,1,1927/28,Title,UNIQUE AND ARTISTIC PICTURE,UNIQUE AND ARTISTIC PICTURE,an0051247,Sunrise,tt0018455,Fox,Fox,co0028775,True,NaN,NaN,NaN,NaN,tt0018455,"Drama,Romance",8.1
2,1,1927/28,Writing,WRITING (Adapted Screenplay),WRITING (Adaptation),an0051263,7th Heaven,tt0018379,Benjamin Glazer,Benjamin Glazer,nm0322227,True,NaN,NaN,NaN,NaN,tt0018379,"Drama,Romance",7.5
3,1,1927/28,Writing,WRITING (Original Story),WRITING (Original Story),an0051266,Underworld,tt0018526,Ben Hecht,Ben Hecht,nm0372942,True,NaN,NaN,NaN,NaN,tt0018526,"Crime,Drama,Film-Noir",7.5
4,1,1927/28,Writing,WRITING (Title Writing),WRITING (Title Writing),an0051269,NaN,NaN,Joseph Farnham,Joseph Farnham,nm0267868,True,NaN,NOTE: This award was not associated with any s...,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
783,97,2024,Title,INTERNATIONAL FEATURE FILM,INTERNATIONAL FEATURE FILM,fake_nomid115,I'm Still Here,tt14961016,Brazil,NaN,NaN,True,NaN,NaN,NaN,NaN,tt14961016,"Biography,Drama,History",8.1
784,97,2024,Title,BEST PICTURE,BEST PICTURE,fake_nomid002,Anora,tt28607951,"Alex Coco, Samantha Quan and Sean Baker, Produ...","Alex Coco, Samantha Quan, Sean Baker","nm7866544,nm1013476,nm0048918",True,NaN,NaN,NaN,NaN,tt28607951,"Comedy,Drama,Romance",7.4
785,97,2024,Title,SHORT FILM (Live Action),SHORT FILM (Live Action),fake_nomid109,I'm Not a Robot,tt19837932,Victoria Warmerdam and Trent,"Victoria Warmerdam, Trent","nm6255610,nm0686692",True,NaN,NaN,NaN,NaN,tt19837932,"Comedy,Drama,Fantasy",7.1
786,97,2024,Writing,WRITING (Adapted Screenplay),WRITING (Adapted Screenplay),fake_nomid040,Conclave,tt20215234,Screenplay by Peter Straughan,Peter Straughan,nm1661186,True,NaN,NaN,NaN,NaN,tt20215234,"Drama,Thriller",7.4


## Filter and Clean the Merged Dataset ##

In [6]:
winning_films_in_range = winning_films_with_genre[~winning_films_with_genre['Year'].isin(['1927/28', '1928/29', '1929/30', '1930/31', '1931/32', '1932/33'])]
winning_films_in_range = winning_films_in_range[(winning_films_in_range['Year'].astype(int) >= 2004) & (winning_films_in_range['Year'].astype(int) <= 2024)]

cleaned_winners = winning_films_in_range.drop_duplicates(subset=['Film'], keep='first').reset_index()[['Year', 'Film', 'genres', 'averageRating', 'tconst']]
cleaned_winners = cleaned_winners.dropna()
cleaned_winners

,Year,Film,genres,averageRating,tconst
0,2004,The Incredibles,"Action,Adventure,Animation",8.0,tt0317705
1,2004,Born into Brothels,"Biography,Documentary,News",7.2,tt0388789
2,2004,Mighty Times: The Children's March,"Documentary,Drama,Short",7.7,tt0443587
3,2004,The Sea Inside,"Biography,Drama",7.9,tt0369702
4,2004,Million Dollar Baby,"Drama,Sport",8.1,tt0405159
...,...,...,...,...,...
167,2024,The Only Girl in the Orchestra,"Documentary,Music,Short",6.6,tt29497240
168,2024,I'm Still Here,"Biography,Drama,History",8.1,tt14961016
169,2024,Anora,"Comedy,Drama,Romance",7.4,tt28607951
170,2024,I'm Not a Robot,"Comedy,Drama,Fantasy",7.1,tt19837932


## Filter and Randomly Sample Non-Winners ##

In [7]:
filtered_imdb = imdb_basics[~imdb_basics['tconst'].isin(cleaned_winners['tconst'].tolist())]
filtered_imdb = filtered_imdb[(filtered_imdb['startYear'] >= 2004) & (filtered_imdb['startYear'] <= 2024)]
filtered_imdb = filtered_imdb[filtered_imdb['titleType'].isin(['movie', 'short'])]

filtered_imdb = filtered_imdb.merge(
    imdb_ratings[['tconst', 'averageRating']],
    on='tconst',
    how='left'
)

filtered_imdb = filtered_imdb[~filtered_imdb['averageRating'].isna()]
filtered_imdb

,tconst,titleType,primaryTitle,originalTitle,isAdult,startYear,endYear,runtimeMinutes,genres,averageRating
1,tt0050396,short,Final Curtain,Final Curtain,0,2012.0,NaN,22,"Horror,Short",4.4
2,tt0056840,short,Aufsätze,Aufsätze,0,2021.0,NaN,10,Short,6.9
3,tt0057369,short,Number 14: Late Superimpositions,Number 14: Late Superimpositions,0,2023.0,NaN,30,Short,5.8
4,tt0060361,short,EMS nr 1,EMS nr 1,0,2016.0,NaN,14,Short,6.5
6,tt0062336,movie,The Tango of the Widower and Its Distorting Mi...,El tango del viudo y su espejo deformante,0,2020.0,NaN,70,Drama,6.4
...,...,...,...,...,...,...,...,...,...,...
1107623,tt9916538,movie,Kuambil Lagi Hatiku,Kuambil Lagi Hatiku,0,2019.0,NaN,123,Drama,7.6
1107624,tt9916544,short,My Sweet Prince,My Sweet Prince,0,2019.0,NaN,12,"Drama,Short",6.8
1107633,tt9916706,movie,Dankyavar Danka,Dankyavar Danka,0,2013.0,NaN,NaN,Comedy,7.7
1107634,tt9916724,short,Hay Que Ser Paciente,Hay Que Ser Paciente,0,2015.0,NaN,3,"Documentary,Short",6.6


In [8]:
random_nonwinners = filtered_imdb.sample(n=1000, replace=False)
random_nonwinners = random_nonwinners.reset_index()
random_nonwinners

,index,tconst,titleType,primaryTitle,originalTitle,isAdult,startYear,endYear,runtimeMinutes,genres,averageRating
0,4638,tt0415141,movie,Mama's Boy,Mama's Boy,0,2007.0,NaN,93,"Comedy,Drama",5.2
1,414521,tt2119482,short,Nickelback Responds to NFL Petition,Nickelback Responds to NFL Petition,0,2011.0,NaN,3,"Comedy,Short",6.1
2,778382,tt3877718,movie,Tokyo Fiancée,Tokyo Fiancée,0,2014.0,NaN,100,"Comedy,Drama,Romance",6.4
3,591482,tt29609813,movie,My Private Line to God,My Private Line to God,0,2024.0,NaN,82,"Drama,Family",6.6
4,292067,tt15740560,movie,In Lichtgeschwindigkeit zum Impfstoff,In Lichtgeschwindigkeit zum Impfstoff,0,2021.0,NaN,52,Documentary,7.4
...,...,...,...,...,...,...,...,...,...,...,...
995,178529,tt13408950,movie,Finding Carlos,Finding Carlos,0,2022.0,NaN,85,"Drama,Family,Music",4.8
996,761049,tt3758228,short,Chiaroscuro,Chiaroscuro,0,2015.0,NaN,8,"Action,Animation,Short",6.7
997,116653,tt11906012,movie,Underplayed,Underplayed,0,2020.0,NaN,87,"Documentary,Music",6.7
998,431052,tt2177827,movie,The Search,The Search,0,2014.0,NaN,135,"Drama,War",6.8


In [9]:
random_nonwinners = random_nonwinners[['startYear', 'primaryTitle', 'genres', 'averageRating']]
random_nonwinners['startYear'] = random_nonwinners['startYear'].astype(int)
random_nonwinners.columns = ['Year', 'Film', 'genres', 'averageRating']
random_nonwinners

/tmp/ipykernel_350/113183732.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  random_nonwinners['startYear'] = random_nonwinners['startYear'].astype(int)


,Year,Film,genres,averageRating
0,2007,Mama's Boy,"Comedy,Drama",5.2
1,2011,Nickelback Responds to NFL Petition,"Comedy,Short",6.1
2,2014,Tokyo Fiancée,"Comedy,Drama,Romance",6.4
3,2024,My Private Line to God,"Drama,Family",6.6
4,2021,In Lichtgeschwindigkeit zum Impfstoff,Documentary,7.4
...,...,...,...,...
995,2022,Finding Carlos,"Drama,Family,Music",4.8
996,2015,Chiaroscuro,"Action,Animation,Short",6.7
997,2020,Underplayed,"Documentary,Music",6.7
998,2014,The Search,"Drama,War",6.8


In [10]:
cleaned_winners = cleaned_winners[['Year', 'Film', 'genres', 'averageRating']]

cleaned_winners['winner'] = [True] * cleaned_winners.shape[0]
random_nonwinners['winner'] = [False] * random_nonwinners.shape[0]

combined_df = pd.concat([cleaned_winners, random_nonwinners])
combined_df

/tmp/ipykernel_350/2657320725.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  random_nonwinners['winner'] = [False] * random_nonwinners.shape[0]


,Year,Film,genres,averageRating,winner
0,2004,The Incredibles,"Action,Adventure,Animation",8.0,True
1,2004,Born into Brothels,"Biography,Documentary,News",7.2,True
2,2004,Mighty Times: The Children's March,"Documentary,Drama,Short",7.7,True
3,2004,The Sea Inside,"Biography,Drama",7.9,True
4,2004,Million Dollar Baby,"Drama,Sport",8.1,True
...,...,...,...,...,...
995,2022,Finding Carlos,"Drama,Family,Music",4.8,False
996,2015,Chiaroscuro,"Action,Animation,Short",6.7,False
997,2020,Underplayed,"Documentary,Music",6.7,False
998,2014,The Search,"Drama,War",6.8,False
